In [ ]:
import sys
!{sys.executable} -m pip install sklearn

In [ ]:
import os
import re
import pickle
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
# ─────────────────────────────────────────
# Paths
# ─────────────────────────────────────────

CLEAN_FILE  = "data/clean/jobs_clean.csv"
MODEL_DIR   = "model"
MODEL_FILE  = os.path.join(MODEL_DIR, "salary_model.pkl")
ENCODER_FILE = os.path.join(MODEL_DIR, "encoders.pkl")
META_FILE   = os.path.join(MODEL_DIR, "model_meta.pkl")

In [ ]:
# ─────────────────────────────────────────
# Role grouping — collapse 1,025 titles
# into ~20 meaningful categories
# ─────────────────────────────────────────

ROLE_GROUPS = {
    "Engineering & Tech":   ["engineer", "developer", "programmer", "software",
                              "devops", "data", "it ", "network", "system",
                              "infrastructure", "backend", "frontend", "fullstack"],
    "Sales & Marketing":    ["sales", "marketing", "business development",
                              "growth", "brand", "digital marketing", "seo",
                              "social media", "commercial"],
    "Finance & Accounting": ["accountant", "accounting", "finance", "auditor",
                              "audit", "tax", "treasury", "financial"],
    "Management":           ["manager", "director", "head of", "chief",
                              "president", "ceo", "coo", "cfo", "vp ",
                              "lead", "supervisor", "coordinator"],
    "Human Resources":      ["hr ", "human resource", "recruitment", "talent",
                              "people operations", "learning and development"],
    "Operations & Logistics":["operations", "logistics", "supply chain",
                               "procurement", "warehouse", "driver", "dispatch"],
    "Healthcare":           ["nurse", "doctor", "physician", "pharmacist",
                              "health", "medical", "clinical", "laboratory",
                              "radiologist", "dentist", "midwife"],
    "Legal":                ["lawyer", "legal", "solicitor", "counsel",
                              "compliance", "paralegal"],
    "Education & Training": ["teacher", "tutor", "lecturer", "trainer",
                              "instructor", "education", "academic"],
    "Customer Service":     ["customer", "client", "support", "helpdesk",
                              "call centre", "customer care"],
    "Admin & Secretarial":  ["admin", "secretary", "receptionist", "office",
                              "clerical", "personal assistant", "pa "],
    "Creative & Design":    ["designer", "creative", "graphic", "ui/ux",
                              "artist", "animator", "photographer", "content"],
    "NGO & Humanitarian":   ["ngo", "humanitarian", "program officer",
                              "field officer", "community", "social worker",
                              "protection", "wash", "nutrition", "caseworker"],
    "Research & Analytics": ["analyst", "research", "data analyst",
                              "business analyst", "insights", "intelligence"],
    "Construction & Estate":["architect", "civil", "structural", "estate",
                              "property", "real estate", "quantity surveyor",
                              "site engineer"],
}


def group_title(title):
    """Map a raw job title to a role group."""
    title_lower = str(title).lower()
    for group, keywords in ROLE_GROUPS.items():
        if any(kw in title_lower for kw in keywords):
            return group
    return "Other"

In [ ]:
# ─────────────────────────────────────────
# Location simplification
# ─────────────────────────────────────────

MAJOR_CITIES = [
    "Lagos", "Abuja", "Port Harcourt", "Kano", "Ibadan", "Remote"
]

def simplify_location(location):
    if pd.isna(location) or location in ["Unknown", ""]:
        return "Other"
    for city in MAJOR_CITIES:
        if city.lower() in str(location).lower():
            return city
    return "Other States"

In [ ]:
# ─────────────────────────────────────────
# Feature engineering
# ─────────────────────────────────────────

def engineer_features(df):
    """Build the feature matrix from clean data."""

    # Work only with rows that have salary
    df = df[df["avg_salary"].notna()].copy()
    print(f"Rows with salary data: {len(df)}")

    # Remove extreme outliers (salaries below 10k or above 10M monthly)
    df = df[(df["avg_salary"] >= 10_000) & (df["avg_salary"] <= 10_000_000)]
    print(f"Rows after outlier removal: {len(df)}")

    # Group titles into role categories
    df["role_group"] = df["title"].apply(group_title)

    # Simplify locations
    df["city"] = df["location"].apply(simplify_location)

    # Clean job_type
    df["job_type"] = df["job_type"].fillna("").str.lower().str.strip()
    df["job_type"] = df["job_type"].apply(
        lambda x: "Full Time" if "full" in x
        else "Part Time" if "part" in x
        else "Contract" if "contract" in x
        else "Other"
    )

    # Clean sector
    df["sector"] = df["sector"].fillna("Unknown")
    df["sector"] = df["sector"].apply(
        lambda x: x if x != "" else "Unknown"
    )

    print(f"\nRole group distribution:")
    print(df["role_group"].value_counts().to_string())
    print(f"\nCity distribution:")
    print(df["city"].value_counts().to_string())

    return df

In [ ]:
# ─────────────────────────────────────────
# Encode + train
# ─────────────────────────────────────────

def train_model(df):
    """Encode features, train model, evaluate."""

    features = ["role_group", "city", "job_type", "sector"]
    target   = "avg_salary"

    # Label encode categorical features
    encoders = {}
    X = pd.DataFrame()

    for col in features:
        le = LabelEncoder()
        X[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le

    y = df[target]

    # Train / test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print(f"\nTraining on {len(X_train)} rows, testing on {len(X_test)} rows")

    # ── Random Forest ──
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)
    rf_mae   = mean_absolute_error(y_test, rf_preds)
    rf_r2    = r2_score(y_test, rf_preds)

    # ── Gradient Boosting ──
    gb = GradientBoostingRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    )
    gb.fit(X_train, y_train)
    gb_preds = gb.predict(X_test)
    gb_mae   = mean_absolute_error(y_test, gb_preds)
    gb_r2    = r2_score(y_test, gb_preds)

    print(f"\n{'─' * 40}")
    print(f"  Model Evaluation")
    print(f"{'─' * 40}")
    print(f"  Random Forest")
    print(f"    MAE : ₦{rf_mae:,.0f}")
    print(f"    R²  : {rf_r2:.3f}")
    print(f"  Gradient Boosting")
    print(f"    MAE : ₦{gb_mae:,.0f}")
    print(f"    R²  : {gb_r2:.3f}")

    # Pick the better model
    if rf_mae <= gb_mae:
        best_model = rf
        best_name  = "Random Forest"
        best_mae   = rf_mae
        best_r2    = rf_r2
    else:
        best_model = gb
        best_name  = "Gradient Boosting"
        best_mae   = gb_mae
        best_r2    = gb_r2

    print(f"\n  Best model: {best_name}")
    print(f"{'─' * 40}")

    # Feature importance
    print(f"\nFeature importances ({best_name}):")
    importances = pd.Series(
        best_model.feature_importances_, index=features
    ).sort_values(ascending=False)
    print(importances.to_string())

    return best_model, encoders, best_name, best_mae, best_r2

In [ ]:
# ─────────────────────────────────────────
# Prediction helper (used by dashboard)
# ─────────────────────────────────────────

def predict_salary(model, encoders, role_group, city, job_type, sector):
    """Predict salary for a given combination of features.
    Returns (min_estimate, max_estimate) in NGN.
    """
    row = {}
    for col, value in zip(
        ["role_group", "city", "job_type", "sector"],
        [role_group, city, job_type, sector]
    ):
        le = encoders[col]
        if value in le.classes_:
            row[col] = le.transform([value])[0]
        else:
            # Unseen label — use most common class
            row[col] = 0

    X = pd.DataFrame([row])
    prediction = model.predict(X)[0]

    # Return a ±15% confidence range
    return round(prediction * 0.85), round(prediction * 1.15)

In [ ]:
# ─────────────────────────────────────────
# Save model
# ─────────────────────────────────────────

def save_model(model, encoders, meta):
    os.makedirs(MODEL_DIR, exist_ok=True)
    with open(MODEL_FILE,   "wb") as f: pickle.dump(model,    f)
    with open(ENCODER_FILE, "wb") as f: pickle.dump(encoders, f)
    with open(META_FILE,    "wb") as f: pickle.dump(meta,     f)
    print(f"\nModel saved to {MODEL_FILE}")
    print(f"Encoders saved to {ENCODER_FILE}")

In [ ]:
# ─────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 50)
    print("  Nigerian Job Market — Salary Predictor")
    print("=" * 50)

    # Load clean data
    print(f"\nLoading {CLEAN_FILE}...")
    df = pd.read_csv(CLEAN_FILE)
    print(f"Loaded {len(df):,} rows")

    # Feature engineering
    print("\nEngineering features...")
    df = engineer_features(df)

    # Train
    print("\nTraining models...")
    model, encoders, model_name, mae, r2 = train_model(df)

    # Save
    meta = {
        "model_name":  model_name,
        "mae":         mae,
        "r2":          r2,
        "role_groups": list(ROLE_GROUPS.keys()) + ["Other"],
        "cities":      MAJOR_CITIES + ["Other States"],
        "job_types":   ["Full Time", "Part Time", "Contract", "Other"],
    }
    save_model(model, encoders, meta)

    # Quick sanity check
    print("\nSanity check — sample predictions:")
    checks = [
        ("Engineering & Tech", "Lagos",  "Full Time", "Information Technology"),
        ("Finance & Accounting", "Abuja", "Full Time", "Accounting, Auditing & Finance"),
        ("Sales & Marketing",  "Lagos",  "Full Time", "Sales"),
        ("NGO & Humanitarian", "Abuja",  "Contract",  "NGO / Humanitarian"),
        ("Management",         "Lagos",  "Full Time", "Management & Business Development"),
    ]
    for role, city, jtype, sector in checks:
        lo, hi = predict_salary(model, encoders, role, city, jtype, sector)
        print(f"  {role} | {city} | {jtype}")
        print(f"    → ₦{lo:,.0f} – ₦{hi:,.0f}/month")